In this tutorial MAST API is used to filter and query different parts of sky and download the related image products. Then, it retrieves World Coordinate System (WCS) information from the downloaded products and provides these as inputs for catelog queries. 

After this step Gaia(or any Vizier catelog) is queried according to a rectangle specified to either focus on a part of the image, span the image, or put the image in a wider context of cateloged objects. Finally both the image and catelog sources will be plotted on the same figure. 

In [1]:
# astroquery.mast.Observations enables us to download images from MAST

from astroquery.mast import(
    Observations, 
)
from astroquery.gaia import Gaia
from astroquery.vizier import Vizier

from astropy.wcs import WCS
from astropy.io import fits
from astropy.coordinates import SkyCoord

import astropy.units as u

import matplotlib.pyplot as plt



Querying and Downloading MAST data

In [4]:
print(Observations.list_missions())
print(len(Observations.list_missions()))


['BEFS', 'EUVE', 'FIMS-SPEAR', 'FUSE', 'GALEX', 'HLA', 'HLSP', 'HST', 'HUT', 'IUE', 'JWST', 'K2', 'K2FFI', 'Kepler', 'KeplerFFI', 'OPO', 'PS1', 'SDSS', 'SPITZER_SHA', 'SWIFT', 'TESS', 'TUES', 'WUPPE']
23


In [5]:
obs_table= Observations.query_criteria(
    dataproduct_type=["image"],
    obs_collection=["JWST"],
    s_dec=[-45.55, -40.45],
    s_ra=[10.0,20.0]  
)

In [7]:


products_by_obs= Observations.get_product_list(obs_table)
print(products_by_obs)

  obsID   obs_collection dataproduct_type ... calib_level   filters   
--------- -------------- ---------------- ... ----------- ------------
228645278           JWST            image ...           1 OPAQUE;G235M
228645278           JWST            image ...           2 OPAQUE;G235M
228645278           JWST            image ...           2 OPAQUE;G235M
228645278           JWST            image ...           1 OPAQUE;G235M
228645278           JWST            image ...           2 OPAQUE;G235M
228645278           JWST            image ...           2 OPAQUE;G235M
228645278           JWST            image ...           2 OPAQUE;G235M
228645278           JWST            image ...           2 OPAQUE;G235M
228645278           JWST            image ...           2 OPAQUE;G235M
228645278           JWST            image ...           2 OPAQUE;G235M
      ...            ...              ... ...         ...          ...
228686637           JWST            image ...           2       F150W2
228686

In [9]:
prods= Observations.filter_products(
products_by_obs,
calib_level=["3"],
productType=["SCIENCE"],
extension="fits",
obsID=["228645392"]
)

prods

obsID,obs_collection,dataproduct_type,obs_id,description,type,dataURI,productType,productGroupDescription,productSubGroupDescription,productDocumentationURL,project,prvversion,proposal_id,productFilename,size,parent_obsid,dataRights,calib_level,filters
str9,str4,str5,str37,str64,str1,str75,str9,str28,str11,str1,str7,str5,str4,str57,int64,str9,str6,int64,str12
228645392,JWST,image,jw05594-o145_t147_nircam_clear-f322w2,exposure/target (L2b/L3): rectified 2D image,D,mast:JWST/product/jw05594-o145_t147_nircam_clear-f322w2_i2d.fits,SCIENCE,Minimum Recommended Products,I2D,--,CALJWST,2.0.1,5594,jw05594-o145_t147_nircam_clear-f322w2_i2d.fits,143127360,228645392,PUBLIC,3,F322W2


In [10]:
# Downloading the data products

manifest= Observations.download_products(prods[0])
print(manifest)

                                               Local Path                                                ...
-------------------------------------------------------------------------------------------------------- ...
./mastDownload/JWST/jw05594-o145_t147_nircam_clear-f322w2/jw05594-o145_t147_nircam_clear-f322w2_i2d.fits ...


In [11]:
# Opening the fits file

fits_file = manifest["Local Path"][0]
print(fits_file)

./mastDownload/JWST/jw05594-o145_t147_nircam_clear-f322w2/jw05594-o145_t147_nircam_clear-f322w2_i2d.fits


The data of interest is usually in hdu['SCI'].data. For example, in JWST and HST files, hdu[1].header will have WCS information, which can be retrieved via astropy.wcs.WCS(hdu[1].header)


In [12]:
with fits.open(fits_file, ignore_missing_simple=True) as hdu:
    image_data = hdu["SCI"].data  # this is a numpy nd array
    wcs_helix = WCS(hdu[1].header) # WCS object from FITS
    

Set DATE-AVG to '2024-08-19T22:01:42.243' from MJD-AVG.
Set DATE-END to '2024-08-19T22:20:35.004' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -21.231906 from OBSGEO-[XYZ].
Set OBSGEO-H to 1501193284.090 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Working with Celestial Coordinates

pixel: part of a detector corresponding to a real 2D area

voxel: A 3D counterpart of pixel, indicating "volume"

spaxel: spatial or spectral pixel

CRVALn: Actual coordinate of the information present in the pixel/voxel

CRPIXn: Location in terms of pixel order, CRPIX1=1 and CRPIX2=1 represents the reference in a 2D image

CDELTn: Delta (coordinate difference) of distance from one pixel to its neighbor

CTYPEn: Eight-char string for axis type, e.g., 'RA---TAN' 'DEC--TAN'

CUNITn: String for unit, default is degree

NAXISn: Number of pixels in each axes

Querying catelogs according to the FITS file coordinate span:

In [13]:
coord = SkyCoord(
    ra= wcs_helix.wcs.crval[0], dec= wcs_helix.wcs.crval[1], unit="deg", frame="icrs"
)
exp_coeff = 0.4 # A multiplier of width and height for querying, either focusing on the image's specific 
#part or putting the image on the big picture

width = u.Quantity(0.1 * exp_coeff, u.deg)
height = u.Quantity(0.1 * exp_coeff, u.deg)


In [ ]:
print("|||| ---- |||| Querying GAIA Data Release III |||| ---- ||||")

# Default GAIA data source, Gaia.MAIN_GAIA_TABLE, is Gaia DR III.

r = Gaia.query_object_async(coordinate=coord, width=width, height=height)
r.pprint(max_lines=12, max_width=130)

In [ ]:
catelog_2mass = "II/246/out"
Vizier.ROW_LIMIT = -1  # alters the 50 rows fdefault limit
r2= Vizier.query_reggion(coord, width=width, catelog=catelog_2mass)
print(r2)
print(
    r2[catelog_2mass].columns
)

In [ ]:
# Differing Column names

ralist= r2[catelog_2mass["RAJ2000"]]
declist= r2[catelog_2mass]["DEJ2000"]
names= r2[catelog_2mass]["2MASS"]


In [ ]:
# Overplotting sources onto the Image
# We can take advantage of different axis objects on a single figure using 
# matplotlib.plt.subplot(projection="") which permits the use of multiple coordinate frames
# on the same plot

plt.figure(figsize=(15,15))
ax= plt.subplot(projection=wcs_helix) # subplot with specific projection

# Showing the image

ax.imshow(image_data, origin="lower",cmap="afmhot",vmax=6)
ax.set_x_label("Right Ascension D.M.S(J2000)", color="cyan")
ax.set_ylabel("Declination D.M.S. (J2000)", color="cyan")
ax.grid(color="cyan",ls="dashed")
ax.tick_params(axis="both",color="cyan",labelcolor="cyan")

# Overlaying galactic coordinates on counter axes

overlay = ax.get_coords_overlay("galactic")
overlay.grid(color="red",ls="dotted")
overlay[0].set_axislabel("Galactic Longitude", color="red")
overlay[1].set_axislabel("Galactic Latitude", color="red")
overlay[0].set_ticks(color="red")
overlay[1].set_ticks(color="red")
overlay[0].set_ticklabel(color="red")
overlay[1].set_ticklabel(color="red")

ax.scatter(ralist, declist, transform=ax.get_transform("icrs"),s=15, edgecolor="red")

for xi, yi, ni in zip(ralist, declist, names):
    plt.text(
        xi,
        yi,
        ni,
        va="bottom",
        ha="center",
        fontsize=7,
        color="blue",
        transform=ax.get_transform("icrs"),
    )
plt.show()